In [4]:
import pickle
import numpy as np
import torch
import gc
import pandas as pd
import scipy.sparse as sp
from scipy.io import loadmat
from sklearn.utils import shuffle
from tqdm import tqdm
def extract_subgraph(edge_index, mask_slc, node_feat=None, node_label=None, 
                     node_timestamp=None, edge_type=None, edge_timestamp=None):
    """
    Extract a subgraph: filter edges, reindex node IDs, and extract edge attributes
    
    Parameters:
        edge_index: (num_edges, 2) edge indices
        mask_slc: IDs of retained nodes (original order defines new IDs)
        node_feat: (num_nodes, feat_dim) node features (optional)
        node_label: (num_nodes,) node labels (optional)
        node_timestamp: (num_nodes,) node timestamps (optional)
        edge_type: (num_edges,) edge types (optional)
        edge_attr: (num_edges, attr_dim) edge attributes (optional)
    
    Returns:
        new_edge_index: (num_filtered_edges, 2) reindexed edges
        new_node_feat: subgraph node features (optional)
        new_node_label: subgraph node labels (optional)
        new_node_timestamp: subgraph node timestamps (optional)
        new_edge_type: subgraph edge types (optional)
        new_edge_attr: subgraph edge attributes (optional)
        node_map: mapping from old IDs to new IDs
    """
    # ========== Step 1: Filter edges ==========
    mask_set = set(mask_slc)
    mask = np.array([u in mask_set and v in mask_set for u, v in edge_index])
    filtered_edges = edge_index[mask]
    
    # ========== Step 2: Filter edge attributes ==========
    new_edge_type = None
    if edge_type is not None:
        new_edge_type = edge_type[mask]
    
    new_edge_timestamp = None
    if edge_timestamp is not None:
        new_edge_timestamp = edge_timestamp[mask]
    
    # ========== Step 3: Reindex node IDs (preserve mask_slc order) ==========
    node_map = {old: new for new, old in enumerate(mask_slc)}
    new_edge_index = np.vectorize(node_map.get)(filtered_edges)
    
    # ========== Step 4: Extract node-related data ==========
    new_node_feat = None
    if node_feat is not None:
        new_node_feat = node_feat[mask_slc]

    new_node_label = None
    if node_label is not None:
        new_node_label = node_label[mask_slc]

    new_node_timestamp = None
    if node_timestamp is not None:
        new_node_timestamp = node_timestamp[mask_slc]
    
    return (new_edge_index, new_node_feat, new_node_label, new_edge_timestamp,
            new_edge_type, node_map)


In [6]:
data_np = np.load('../data/tfinance.npz')
print([f for f in data_np.keys()])

x = data_np['x']
y = data_np['y']
train_mask = data_np['train_mask']
valid_mask = data_np['valid_mask']
test_mask = data_np['test_mask']

edge_type = data_np['edge_type']
edge_timestamp = data_np['edge_timestamp']
edge_index = data_np['edge_index']


del data_np
[gc.collect() for _ in range(5)]
# mem_gib = psutil.Process(os.getpid()).memory_info().rss / 1024**3
# print(f"Current process memory: {mem_gib:.2f} GiB")

['x', 'y', 'train_mask', 'valid_mask', 'test_mask', 'edge_index', 'edge_type', 'edge_timestamp']


[581, 0, 0, 0, 0]

In [7]:
# ---------- 5. Export ----------
mask_slc_lst = [train_mask, valid_mask, test_mask]
type_lst = ['train', 'valid', 'test']

for i in tqdm(range(len(type_lst)) ):
    # mask_slc = train_mask
    
    # Use
    edge_index_slc, x_slc, y_slc, edge_timestamp_slc, edge_type_slc, mapping = extract_subgraph(
        edge_index, mask_slc_lst[i]
        ,node_feat=x
        ,node_label=y
        ,edge_type=edge_type
        ,edge_timestamp = edge_timestamp
    )
    
    
    # ---------- 5. Export ----------
    out_file = '../data_split/tfinance_{}.npz'.format(type_lst[i])
    np.savez(out_file,
             x=x_slc,
             y=y_slc,
             edge_index=edge_index_slc,
             edge_type=edge_type_slc,
             edge_timestamp=edge_timestamp_slc)
    
    print('✅ Exported', out_file)
    print('Shape check: x={}, y={}, edge_index={}, edge_type={}'.format(
        x_slc.shape, y_slc.shape, edge_index_slc.shape, edge_type_slc.shape))

 33%|███▎      | 1/3 [01:02<02:04, 62.15s/it]

✅ Exported ../data_split/tfinance_train.npz
Shape check: x=(23614, 10), y=(23614,), edge_index=(15775696, 2), edge_type=(15775696,)


 67%|██████▋   | 2/3 [01:49<00:53, 53.45s/it]

✅ Exported ../data_split/tfinance_valid.npz
Shape check: x=(7871, 10), y=(7871,), edge_index=(1668186, 2), edge_type=(1668186,)


100%|██████████| 3/3 [02:37<00:00, 52.39s/it]

✅ Exported ../data_split/tfinance_test.npz
Shape check: x=(7872, 10), y=(7872,), edge_index=(1564284, 2), edge_type=(1564284,)
